### RAG Pipelines- Data Ingestion to Vector DB Pipeline

## Steps 
#### convert to Documents -> break into chunks -> embedding -> vector store

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\user\AppData\Local\Temp\ipykernel_2792\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\user\Desktop\AgenticAI\RAG-Tutorials\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents) # our documnet is also list and all_documents is also a list we use append then all_documents will be a list of list so we use extend to make it a single list.
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 15 PDF files to process

Processing: Almas Aslam CV 2026.pdf
  ✓ Loaded 1 pages

Processing: Aman Ghaus (Resume).pdf
  ✓ Loaded 1 pages

Processing: AMINA OVAIS_Biodata (1).pdf
  ✓ Loaded 1 pages

Processing: Arshala Firoz.pdf
  ✓ Loaded 1 pages

Processing: Faizul_Bari_Compliance_Audit_CV_Final_NoSpace.pdf
  ✓ Loaded 1 pages

Processing: intekhab_software_dev_resume (1).pdf
  ✓ Loaded 1 pages

Processing: Rahim_ResumeLatest.pdf
  ✓ Loaded 1 pages

Processing: RESUME_2025_Frontend_Developer_cv.pdf
  ✓ Loaded 2 pages

Processing: Shahbaz_Alam_RESUMEE.pdf
  ✓ Loaded 2 pages

Processing: Shamshad_Alam_Premium_Resume (1).pdf
  ✓ Loaded 1 pages

Processing: Sharique_Ahmad_CV1.pdf
  ✓ Loaded 1 pages

Processing: Tashleem Quraishi_Offcampus CV Final (1) (1).pdf
  ✓ Loaded 1 pages

Processing: Tazeen_resume_pre (1).pdf
  ✓ Loaded 2 pages

Processing: Tazeen_resume_pre.pdf
  ✓ Loaded 2 pages

Processing: Waqqar_Rashid_Resume.pdf
  ✓ Loaded 2 pages

Total documents loaded: 20


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-02-18T13:02:56+05:30', 'author': '', 'moddate': '2026-02-18T13:02:56+05:30', 'title': 'Microsoft Word - Almas_Aslam_update CV_2026', 'source': '..\\data\\pdf\\Almas Aslam CV 2026.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Almas Aslam CV 2026.pdf', 'file_type': 'pdf'}, page_content='Almas Aslam \n■ +91 9875592403 | ✉■ aaalmas7@gmail.com \n \nProfessional Summary \n \nResults-driven finance and operations professional with expertise in Accounts Payable (AP), Procure-to-Pay \n(PTP), and Order Management using SAP. Skilled in payment processing, vendor management, invoice \nresolution, and financial reporting with proven ability to streamline workflows and ensure compliance. Strong \ninterpersonal and communication skills, adept at cross-functional collaboration, and recognized for accuracy, \nefficiency, and client satisfaction. Quick learner, adaptable to dynami

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") #split document is a list of document so we use split_docs[0] to get the first document and then we use .page_content to get the content of the document and then we use [:200] to get the first 200 characters of the content.
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 20 documents into 67 chunks

Example chunk:
Content: Almas Aslam 
■ +91 9875592403 | ✉■ aaalmas7@gmail.com 
 
Professional Summary 
 
Results-driven finance and operations professional with expertise in Accounts Payable (AP), Procure-to-Pay 
(PTP), and ...
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-02-18T13:02:56+05:30', 'author': '', 'moddate': '2026-02-18T13:02:56+05:30', 'title': 'Microsoft Word - Almas_Aslam_update CV_2026', 'source': '..\\data\\pdf\\Almas Aslam CV 2026.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Almas Aslam CV 2026.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-02-18T13:02:56+05:30', 'author': '', 'moddate': '2026-02-18T13:02:56+05:30', 'title': 'Microsoft Word - Almas_Aslam_update CV_2026', 'source': '..\\data\\pdf\\Almas Aslam CV 2026.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Almas Aslam CV 2026.pdf', 'file_type': 'pdf'}, page_content='Almas Aslam \n■ +91 9875592403 | ✉■ aaalmas7@gmail.com \n \nProfessional Summary \n \nResults-driven finance and operations professional with expertise in Accounts Payable (AP), Procure-to-Pay \n(PTP), and Order Management using SAP. Skilled in payment processing, vendor management, invoice \nresolution, and financial reporting with proven ability to streamline workflows and ensure compliance. Strong \ninterpersonal and communication skills, adept at cross-functional collaboration, and recognized for accuracy, \nefficiency, and client satisfaction. Quick learner, adaptable to dynami

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    # we gave texts to below functions and said le bhai embeddings generate kar de and it will return embeddings in numpy array format.
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager_obj=EmbeddingManager()
embedding_manager_obj


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1987.50it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\user\AppData\Local\Temp\ipykernel_2792\3517151387.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) # here we are making vectorDB client which will interact with VectorDB
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                    }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    # Ab collection ban gaya and client ban gaya ab hume documents ko add karna hai to vector store me so we will write a function for that.
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        # it takes both documents and embeddings and merges them into the vector store.
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): # zip will club the documents and embeddings together and enumerate will give index for both of them combined.
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" #creating a unique id for each document.
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata) # copying the existing metadata from the document and converting it into a dictionary so that we can add more metadata to it.
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore_obj=VectorStore()
vectorstore_obj
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 67


In [9]:
chunks

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-02-18T13:02:56+05:30', 'author': '', 'moddate': '2026-02-18T13:02:56+05:30', 'title': 'Microsoft Word - Almas_Aslam_update CV_2026', 'source': '..\\data\\pdf\\Almas Aslam CV 2026.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Almas Aslam CV 2026.pdf', 'file_type': 'pdf'}, page_content='Almas Aslam \n■ +91 9875592403 | ✉■ aaalmas7@gmail.com \n \nProfessional Summary \n \nResults-driven finance and operations professional with expertise in Accounts Payable (AP), Procure-to-Pay \n(PTP), and Order Management using SAP. Skilled in payment processing, vendor management, invoice \nresolution, and financial reporting with proven ability to streamline workflows and ensure compliance. Strong \ninterpersonal and communication skills, adept at cross-functional collaboration, and recognized for accuracy, \nefficiency, and client satisfaction. Quick learner, adaptable to dynami

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
# above code is a shortcut of 
# texts = []
#for doc in chunks:
#   texts.append(doc.page_content)


## Generate the Embeddings

embeddings=embedding_manager_obj.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore_obj.add_documents(chunks,embeddings)

Generating embeddings for 67 texts...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches: 100%|██████████| 3/3 [00:05<00:00,  1.95s/it]


Generated embeddings with shape: (67, 384)
Adding 67 documents to vector store...
Successfully added 67 documents to vector store
Total documents in collection: 134


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store_obj2: VectorStore, embedding_manager_obj2: EmbeddingManager ):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store_obj2
        self.embedding_manager = embedding_manager_obj2

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0] #coverting query into embedding and [0] is used to get the first element of the list returned by generate_embeddings since we are passing a single query.
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], #currently query embedding is numpy array and here we are converting it to list so that it can be passed to the query function of the vector store.
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]: #results['documents'] check if any documents were returned and results['documents'][0] check if the first element of the list is not empty.
                print(results)
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    print(f"distance {distance}")
                    print(f"similarity_score {similarity_score}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever_obj=RAGRetriever(vectorstore_obj,embedding_manager_obj)



In [12]:
rag_retriever_obj

In [13]:
rag_retriever_obj.retrieve("Skills")

Retrieving documents for query: 'Skills'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.31it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_9e415acc_3', 'doc_f98ec7d0_3', 'doc_f045f1a0_63', 'doc_66da8331_63', 'doc_11092000_62']], 'embeddings': None, 'documents': [['• Proficient in MS Excel, MS Word, and Tally ERP 9 \n• Skilled in computer fundamentals and accounting tools \n \nLeadership & Extra-Curricular Activities \n• Head Boy – Central Model School (2015) \n• Captain – Rajasthan Cricket Club, U-19 Tournament (2017) \n• Managed Acharya Jagadish Chandra Bose College team at Umang Concert (2019) \n• Volunteered as a tutor for school-level students.', '• Proficient in MS Excel, MS Word, and Tally ERP 9 \n• Skilled in computer fundamentals and accounting tools \n \nLeadership & Extra-Curricular Activities \n• Head Boy – Central Model School (2015) \n• Captain – Rajasthan Cricket Club, U-19 Tournament (2017) \n• Managed Acharya Jagadish Chandra Bose College team at Umang Concert (2019) \n• Volunteered as a tutor for school-level students.', 'Compliance, Knowledge Base 

[{'id': 'doc_9e415acc_3',
  'content': '• Proficient in MS Excel, MS Word, and Tally ERP 9 \n• Skilled in computer fundamentals and accounting tools \n \nLeadership & Extra-Curricular Activities \n• Head Boy – Central Model School (2015) \n• Captain – Rajasthan Cricket Club, U-19 Tournament (2017) \n• Managed Acharya Jagadish Chandra Bose College team at Umang Concert (2019) \n• Volunteered as a tutor for school-level students.',
  'metadata': {'page': 0,
   'moddate': '2026-02-18T13:02:56+05:30',
   'total_pages': 1,
   'doc_index': 3,
   'creationdate': '2026-02-18T13:02:56+05:30',
   'author': '',
   'source_file': 'Almas Aslam CV 2026.pdf',
   'content_length': 384,
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\Almas Aslam CV 2026.pdf',
   'page_label': '1',
   'title': 'Microsoft Word - Almas_Aslam_update CV_2026',
   'producer': 'Microsoft: Print To PDF',
   'creator': 'PyPDF'},
  'similarity_score': 0.406730055809021,
  'distance': 0.593269944190979,
  'rank': 1},
 {'id': 

In [14]:
rag_retriever_obj.retrieve("what is their maximum eduction")


Retrieving documents for query: 'what is their maximum eduction'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.84it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_e58b48c4_46', 'doc_2862eb91_46', 'doc_fc6bee80_65', 'doc_54d8d61c_65', 'doc_b06dde83_64']], 'embeddings': None, 'documents': [['and SCM. \n• QA Delivery & Team Leadership: Directed an 11-member QA team through SIT, FAT, and E2E validation cycles. \nStandardized test planning, led daily Defect Triage sessions, and delivered structured Knowledge Transfer (KT) to cross-\nfunctional teams. \n• Quality & Coverage Optimization: Expanded test suite coverage by 60% through automated frameworks. Personally \nidentified and resolved 127 high-priority defects during FAT/SIT cycles (highest individual count on the project). \n• ERP Upgrade & Advisory: Managed business observations converted into Change Requests (CRs) across AP, AR, GL, \nProcurement, and OBIEE. Delivered E2E testing for ERP Lift & Shift, BI Upgrades, and 43 Agile sprints for Edge4Health. \nCERTIFICATIONS & EDUCATION \nCertifications \n• ISTQB Certified Tester Foundation Leve

[{'id': 'doc_e58b48c4_46',
  'content': 'and SCM. \n• QA Delivery & Team Leadership: Directed an 11-member QA team through SIT, FAT, and E2E validation cycles. \nStandardized test planning, led daily Defect Triage sessions, and delivered structured Knowledge Transfer (KT) to cross-\nfunctional teams. \n• Quality & Coverage Optimization: Expanded test suite coverage by 60% through automated frameworks. Personally \nidentified and resolved 127 high-priority defects during FAT/SIT cycles (highest individual count on the project). \n• ERP Upgrade & Advisory: Managed business observations converted into Change Requests (CRs) across AP, AR, GL, \nProcurement, and OBIEE. Delivered E2E testing for ERP Lift & Shift, BI Upgrades, and 43 Agile sprints for Edge4Health. \nCERTIFICATIONS & EDUCATION \nCertifications \n• ISTQB Certified Tester Foundation Level (CTFL) \n• ISTQB Agile Tester Certification \nEducation \n• Bachelor of Technology (B.Tech) – 70.05% \n• Maulana Azad College of Eng. & Tech (

### RAG Pipeline- VectorDB To LLM Output Generation

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [27]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [28]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3.6-27b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"], #these will be variables which will be there in my prompt template.
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context} #this is the context variable which we defined above.

Question: {question} #this is the question variable which we defined above.

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query) #now the variables we defined in prompt_template is getting actual values which are from the function parameters context and query.
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """

        # below is a simple prompt where we are not using any prompt template or initializing any variables. We are just passing the context and query directly to the model.
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [29]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: qwen/qwen3.6-27b
Groq LLM initialized successfully!


In [30]:
### get the context from the retriever and pass it to the LLM

context=rag_retriever_obj.retrieve("What is their highest education level?")
context

Retrieving documents for query: 'What is their highest education level?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.81it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_c93e5c62_39', 'doc_71884463_39', 'doc_fc6bee80_65', 'doc_54d8d61c_65', 'doc_f98ec7d0_3']], 'embeddings': None, 'documents': [['EDUCATION \n.Bachelor of Computer Application (BCA) - Gaya College, Gaya (Magadh \nUniversity) | 71% | 2023   \n.Higher Secondary (XII)-MIRZA GHALIB College | 2020                             \n.Secondary (X)- QUASIMI High School| 2018 \nCERTIFICATIONS \n• Java with DSA - Coding Blocks \n• Core Java Programming – Naresh IT \n• HTML, CSS, JavaScript – Offline Course Naresh IT \nACHIEVEMENTS \n• Solved 50+ problems on LeetCode & HackerRank \n• Strong problem-solving and analytical skills', 'EDUCATION \n.Bachelor of Computer Application (BCA) - Gaya College, Gaya (Magadh \nUniversity) | 71% | 2023   \n.Higher Secondary (XII)-MIRZA GHALIB College | 2020                             \n.Secondary (X)- QUASIMI High School| 2018 \nCERTIFICATIONS \n• Java with DSA - Coding Blocks \n• Core Java Programming – Naresh 

[{'id': 'doc_c93e5c62_39',
  'content': 'EDUCATION \n.Bachelor of Computer Application (BCA) - Gaya College, Gaya (Magadh \nUniversity) | 71% | 2023   \n.Higher Secondary (XII)-MIRZA GHALIB College | 2020                             \n.Secondary (X)- QUASIMI High School| 2018 \nCERTIFICATIONS \n• Java with DSA - Coding Blocks \n• Core Java Programming – Naresh IT \n• HTML, CSS, JavaScript – Offline Course Naresh IT \nACHIEVEMENTS \n• Solved 50+ problems on LeetCode & HackerRank \n• Strong problem-solving and analytical skills',
  'metadata': {'page_label': '2',
   'moddate': '2026-05-19T12:33:16+05:30',
   'subject': '(unspecified)',
   'source_file': 'Shahbaz_Alam_RESUMEE.pdf',
   'producer': 'Microsoft® Word 2024',
   'total_pages': 2,
   'source': '..\\data\\pdf\\Shahbaz_Alam_RESUMEE.pdf',
   'author': '(anonymous)',
   'file_type': 'pdf',
   'creationdate': '2026-05-19T12:33:16+05:30',
   'content_length': 479,
   'title': '(anonymous)',
   'page': 1,
   'doc_index': 39,
   'creato

In [31]:
model_with_rag=GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
model_with_rag.generate_response("What is their highest education level?",context[0]["content"],500)

Initialized Groq LLM with model: qwen/qwen3.6-27b


"\n<think>\nHere's a thinking process:\n\n1.  **Analyze User Input:**\n   - **Context:** Contains education details: Bachelor of Computer Application (BCA) from Gaya College (Magadh University) with 71% in 2023, Higher Secondary (XII) in 2020, Secondary (X) in 2018. Also includes certifications and achievements.\n   - **Question:** What is their highest education level?\n   - **Task:** Provide a clear and informative answer based on the context. If insufficient info, state that.\n\n2.  **Identify Key Information in Context:**\n   - Education section lists:\n     - Bachelor of Computer Application (BCA) - 2023\n     - Higher Secondary (XII) - 2020\n     - Secondary (X) - 2018\n   - The highest level listed is the Bachelor's degree (BCA).\n\n3.  **Formulate Answer:**\n   - Directly answer the question based on the context.\n   - State the highest education level clearly.\n   - Keep it concise.\n   - Draft: Based on the provided context, their highest education level is a Bachelor of Comp

#### If we want to ask What is their highest education level or any question for all candidates selected in a context we can use below code

In [32]:
combined_text = "\n\n".join(
    var["content"] for var in context ## har 2 line break ke baad var["content"] ko append krte gye
)
combined_text

'EDUCATION \n.Bachelor of Computer Application (BCA) - Gaya College, Gaya (Magadh \nUniversity) | 71% | 2023   \n.Higher Secondary (XII)-MIRZA GHALIB College | 2020                             \n.Secondary (X)- QUASIMI High School| 2018 \nCERTIFICATIONS \n• Java with DSA - Coding Blocks \n• Core Java Programming – Naresh IT \n• HTML, CSS, JavaScript – Offline Course Naresh IT \nACHIEVEMENTS \n• Solved 50+ problems on LeetCode & HackerRank \n• Strong problem-solving and analytical skills\n\nEDUCATION \n.Bachelor of Computer Application (BCA) - Gaya College, Gaya (Magadh \nUniversity) | 71% | 2023   \n.Higher Secondary (XII)-MIRZA GHALIB College | 2020                             \n.Secondary (X)- QUASIMI High School| 2018 \nCERTIFICATIONS \n• Java with DSA - Coding Blocks \n• Core Java Programming – Naresh IT \n• HTML, CSS, JavaScript – Offline Course Naresh IT \nACHIEVEMENTS \n• Solved 50+ problems on LeetCode & HackerRank \n• Strong problem-solving and analytical skills\n\n• Higher Se

In [33]:
model_with_rag.generate_response("What is their highest education level?",combined_text,500)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Context:** Contains two distinct resume/profile snippets merged together.\n     - Snippet 1: Education includes Bachelor of Computer Application (BCA) from Gaya College, Magadh University (2023, 71%), Higher Secondary (XII) (2020), Secondary (X) (2018). Certifications in Java, DSA, HTML/CSS/JS. Achievements in LeetCode/HackerRank.\n     - Snippet 2: Education includes Higher Secondary (HS), ISC Board (2020-2022, 58%), Secondary School (10th), ICSE Board (2018-2020, 65%). Certifications from Oracle, HP, Don Bosco. Hobbies/Skills in MS Office, Tally. Leadership roles.\n   - **Question:** "What is their highest education level?"\n   - **Task:** Provide a clear and informative answer based on the context. If insufficient info, state so.\n\n2.  **Identify Key Information in Context:**\n   - The context actually contains *two different* educational backgrounds merged together.\n   - First part: Bachelor of Compute

In [34]:
context2=rag_retriever_obj.retrieve("who has best skill set for AI engineer")
combined_text2 = "\n\n".join(
    var["content"] for var in context2 ## har 2 line break ke baad var["content"] ko append krte gye
)
combined_text2


Retrieving documents for query: 'who has best skill set for AI engineer'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.93it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_e77ae09f_10', 'doc_5b68cb1d_10', 'doc_e6272a4e_40', 'doc_d1803f21_40', 'doc_d28da382_52']], 'embeddings': None, 'documents': [['ARSHALA FIROZ JUNIOR ENGINEER\nCONTACT\n✉ arshuuu1515@gmail.com\n\uf015 Flat no D/5, Gulab bagh col\nony Near amar jyoti school ma\nngo\n\uf095 9142219498\n\uf024 Indian\nOBJECTIVE\nDedicated Computer Science\nengineer with a strong\nfoundation in software\ndevelopment and problem-\nsolving. Seeking to leverage\nexpertise in programming,\nsystem design, and\ncollaborative team\nenvironments to drive\ninnovation and deliver\nimpactful technology solutions\nin a fast-paced organization.\nLANGUAGES\nSKILLS\nSUMMARY\nDedicated professional with a Diploma in Computer Science and a solid foundation in\nCRM. Experienced in leveraging technical knowledge to enhance customer\nrelationships and streamline processes, demonstrating strong problem-solving abilities.\nKnown for exceptional teamwork and leadership skil

'ARSHALA FIROZ JUNIOR ENGINEER\nCONTACT\n✉ arshuuu1515@gmail.com\n\uf015 Flat no D/5, Gulab bagh col\nony Near amar jyoti school ma\nngo\n\uf095 9142219498\n\uf024 Indian\nOBJECTIVE\nDedicated Computer Science\nengineer with a strong\nfoundation in software\ndevelopment and problem-\nsolving. Seeking to leverage\nexpertise in programming,\nsystem design, and\ncollaborative team\nenvironments to drive\ninnovation and deliver\nimpactful technology solutions\nin a fast-paced organization.\nLANGUAGES\nSKILLS\nSUMMARY\nDedicated professional with a Diploma in Computer Science and a solid foundation in\nCRM. Experienced in leveraging technical knowledge to enhance customer\nrelationships and streamline processes, demonstrating strong problem-solving abilities.\nKnown for exceptional teamwork and leadership skills, fostering collaborative\nenvironments that drive project success. Committed to continuous learning and\nprofessional growth, with a strong work ethic that ensures reliability and h

In [35]:
model_with_rag.generate_response("can you tell me who has best skill set for AI engineer?",combined_text2,500)


'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Context:** Contains three resumes/profiles:\n     - Arshala Firoz: Junior Engineer, Computer Science, CRM, software development, problem-solving.\n     - MD. Shamshad Alam: QA Automation Engineer, Java, Selenium, Cucumber, TestNG, Maven, Jenkins, Agile. Experience validating AI/ML datasets, face/image annotation for computer vision.\n     - Tazeen Iffat: Automation Test Engineer, Selenium WebDriver, Java, TestNG, POM, Data-Driven Testing, Agile, SDLC/STLC.\n   - **Question:** "can you tell me who has best skill set for AI engineer?"\n   - **Constraint:** Answer accurately and concisely based *only* on the provided context. If insufficient info, state that.\n\n2.  **Evaluate Context against Question:**\n   - The question asks for the "best skill set for AI engineer" among the three candidates.\n   - Let\'s check each candidate\'s AI-related skills/experience:\n     - *Arshala Firoz:* Computer Science diploma,

### Integration Vectordb Context pipeline With LLM output
##### Here retrieve, comining text and augmented response generation all will be done in 1 function

In [36]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3.6-27b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "" # here we are just joining the content of the retrieved documents with 2 line breaks in between. If no results are found, context will be an empty string.
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [37]:
answer=rag_simple("Who got the overall best skill set for AI engineer?",rag_retriever_obj,llm)
print(answer)

Retrieving documents for query: 'Who got the overall best skill set for AI engineer?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.02it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_e6272a4e_40', 'doc_d1803f21_40', 'doc_e77ae09f_10']], 'embeddings': None, 'documents': [['MD. SHAMSHAD ALAM \nQA AUTOMATION ENGINEER  \nshamshadalam11786@gmail.com   |   +91 98754 23683   |   Bihar, India   |   Open to Relocation \nPROFILE  \nDetail-oriented QA Automation Engineer with hands-on expertise in Java, Selenium WebDriver, Cucumber (BDD), TestNG, Maven, and \nJenkins, backed by solid grounding in Agile delivery. Brings direct experience validating AI/ML datasets and building automation \nframeworks end-to-end, with a strong focus on quality, precision, and continuous improvement. \nCORE TECHNICAL SKILLS  \nJava  Selenium WebDriver  Cucumber (BDD)  TestNG \nMaven  Jenkins (CI/CD)  Git / GitHub  JIRA \nManual Testing  Regression Testing  Agile / Scrum  SDLC / STLC \nPROFESSIONAL EXPERIENCE  \nQA / Manual Test Associate — Globsyn IT Services Pvt. Ltd. Jul 2019 — Mar 2022 \n \n● Performed manual validation and quality check


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** Two resumes/profiles are provided.
     - Profile 1: MD. SHAMSHAD ALAM - QA Automation Engineer. Skills: Java, Selenium, Cucumber, TestNG, Maven, Jenkins, Git, JIRA, Manual/Regression Testing, Agile/Scrum. Experience: QA/Manual Test Associate at Globsyn IT Services (Jul 2019 - Mar 2022). Key experience: "Performed manual validation and quality checks on AI/ML training datasets to ensure data accuracy and integrity." and "Carried out face and image annotation for computer vision model development."
     - Profile 2: ARSHALA FIROZ - Junior Engineer. Skills: Computer Science foundation, CRM, programming, system design. Summary mentions Diploma in CS, CRM experience, problem-solving, teamwork. No specific AI/ML skills or experience mentioned.
   - **Question:** "Who got the overall best skill set for AI engineer?"
   - **Constraint:** "Use the following context to answer the question concisely."

2.  **Evalu

In [38]:
answer=rag_simple("can you summarize Md Faizul Bari resume?",rag_retriever_obj,llm)
print(answer)

Retrieving documents for query: 'can you summarize Md Faizul Bari resume?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.77it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_1c0fd65d_13', 'doc_650585b2_13', 'doc_9cd16c5a_16']], 'embeddings': None, 'documents': [['MD FAIZUL BARI\nPROFESSIONAL SUMMARY\nQuality Analyst / ML Data Associate with 4+ years of experience at Amazon, specializing in compliance-focused quality\nassurance, audit reviews, data validation, process controls, and root cause analysis. Experienced in reviewing high-volume\noperational data for adherence to SOPs, internal guidelines, quality standards, and regulatory requirements. Maintained 99%+ audit\ncompliance with zero critical findings in annual compliance reviews and contributed to a 30%+ reduction in operational data errors\nthrough standardized QA controls and process improvement.\nCORE COMPETENCIES\n\x7f Compliance Monitoring & SOP Adherence\n\x7f Audit Reviews & Audit Support\n\x7f Quality Assurance & Process Controls\n\x7f Data Validation & Verification\n\x7f Risk Identification & Process Gaps\n\x7f Root Cause Analysis (RCA


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** A resume for "MD FAIZUL BARI" containing Professional Summary, Core Competencies, Technical Skills & Tools, Languages, Education, and Personal Details. Note: The context repeats the summary and competencies twice, but that's fine.
   - **Question:** "can you summarize Md Faizul Bari resume?"
   - **Constraint:** "Use the following context to answer the question concisely."

2.  **Identify Key Information from Context:**
   - **Name:** MD Faizul Bari
   - **Role/Experience:** Quality Analyst / ML Data Associate with 4+ years at Amazon.
   - **Key Achievements:** Maintained 99%+ audit compliance with zero critical findings; reduced operational data errors by 30%+; improved team efficiency by 25% through process improvements.
   - **Core Competencies:** Compliance monitoring, audit reviews, QA, data validation, RCA, process improvement, cross-functional collaboration, advanced MS Excel.
   - **Technical Ski

### Enhanced RAG Pipeline Features
##### Nothing just adding meta data in response

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The text describes several hard negative mining techniques used in contrastive learning for retrieval models:

* **ANCE:** Uses asynchronous ANN indexing and checkpoint states to periodically update hard negatives.
* **Conan-Embedding:** Employs a dynamic strategy, excluding and refreshing samples based on score thresholds.
* **NV-Retriever:**  Proposes positive-aware mining with TopK-MarginPos and TopKPercPos filtering to reduce false negatives.
* **LGAI-Embedding:** Builds on NV-Retriever, using ANNA IR as a teacher model to identify high-quality hard negatives and TopKPercPos filtering. 



Sources: [{'source': 'emneddings.pdf', 'page': 4, 'score': 0.18709993362426758, 'preview': 'QZhou-Embedding Technical Report\n Kingsoft AI\n2.4 Hard Negative Mining Techniques\nHard negatives serve as essential components in contrastive lear ning for retrieval model\ntraining. Early work like ANCE[\n46] proposed an asynchronous ANN indexing mech-\nanism that periodically updates hard nega

#### Finally below is how an Output should be:
##### Response + citation(reference) + summary(optional it has bool value user can do true or false)

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 88.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention functi

on can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is all you need

Answer:

Final Answer: "Attention Is All You Need" is a paper that introduced the Transformer model, which relies solely on attention mechanisms for sequence transduction tasks, eliminating the need for recurrent or convolutional networks.  


Citations:
[1] attention.pdf (page 2)
[2] attention.pdf (page 2)
[3] attention.pdf (page 2)
Summary: The paper "Attention Is All You Need" introduced the Transformer model, a novel architecture for sequence transduction tasks.  This model utilizes only attention mechanisms, dispensing with traditional recur